In [61]:
from __future__ import annotations

import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

In [62]:
TRAIN_FEATURE_PATHS = [
    Path("Extracted_Features/BoW_int.npy"),
    Path("Extracted_Features/Normalized_CH.npy"),
    Path("Extracted_Features/Normalized_CM55.npy"),
    Path("Extracted_Features/Normalized_CORR.npy"),
    Path("Extracted_Features/Normalized_EDH.npy"),
    Path("Extracted_Features/Normalized_WT.npy"),
]
TEST_FEATURE_PATHS = [
    Path("Extracted_Features_Test/BoW_int.npy"),
    Path("Extracted_Features_Test/Normalized_CH.npy"),
    Path("Extracted_Features_Test/Normalized_CM55.npy"),
    Path("Extracted_Features_Test/Normalized_CORR.npy"),
    Path("Extracted_Features_Test/Normalized_EDH.npy"),
    Path("Extracted_Features_Test/Normalized_WT.npy"),
]

In [63]:
@dataclass
class TrainConfig:
    batch_size: int = 256
    epochs: int = 35
    lr: float = 1e-3
    weight_decay: float = 1e-4
    val_ratio: float = 0.2
    threshold: float = 0.5
    random_seed: int = 42
    channels: int = 64
    dropout: float = 0.2

In [64]:
class FeatureDataset(Dataset):
    def __init__(self, features: np.ndarray, labels: np.ndarray):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.features[idx], self.labels[idx]

In [65]:
class BasicBlock1D(nn.Module):
    def __init__(self, channels: int, dropout: float):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(self.block(x) + x)

In [66]:
class ResNet6Features(nn.Module):
    """ResNet-6 for concatenated tabular feature vectors.

    Counted as: stem conv + 2 residual blocks * 2 convs + linear head = 6
    trainable layers. The feature vector is treated as a 1D signal.
    """

    def __init__(self, input_dim: int, num_classes: int, channels: int = 64, dropout: float = 0.2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, channels, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(channels),
            nn.ReLU(inplace=True),
        )
        self.layer1 = BasicBlock1D(channels, dropout)
        self.layer2 = BasicBlock1D(channels, dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(channels, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)

In [67]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [68]:
def get_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

In [69]:
def load_array(path: Path) -> np.ndarray:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path.resolve()}")
    return np.load(path).astype(np.float32)

In [70]:
def load_features(data_root: Path, paths: list[Path]) -> np.ndarray:
    arrays = [load_array(data_root / path) for path in paths]
    n_rows = {array.shape[0] for array in arrays}
    if len(n_rows) != 1:
        raise ValueError(f"Feature row counts do not match: {sorted(n_rows)}")
    return np.concatenate(arrays, axis=1)

In [71]:
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, threshold: float) -> dict[str, float]:
    model.eval()
    all_targets, all_probs, all_preds = [], [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            probs = torch.sigmoid(model(x))
            all_probs.append(probs.cpu().numpy())
            all_preds.append((probs > threshold).float().cpu().numpy())
            all_targets.append(y.numpy())

    targets = np.vstack(all_targets)
    probs = np.vstack(all_probs)
    preds = np.vstack(all_preds)
    try:
        map_score = average_precision_score(targets, probs, average="macro")
    except ValueError:
        map_score = 0.0
    return {
        "mAP": float(map_score),
        "micro_f1": float(f1_score(targets, preds, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(targets, preds, average="macro", zero_division=0)),
    }


In [72]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    total_seen = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        total_seen += len(y)

    return total_loss / max(total_seen, 1)

In [73]:
def save_json(path: Path, data: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")

In [ ]:
# Notebook experiment settings
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "dataset").exists() else NOTEBOOK_DIR.parent
DATA_ROOT = PROJECT_ROOT / "dataset"
OUTPUT_DIR = PROJECT_ROOT / "ResNet" / "resnet6_feature_runs"
NUM_WORKERS = 0

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Data root: {DATA_ROOT.resolve()}")

config = TrainConfig(
    batch_size=32,
    epochs=50,
    lr=1e-4,
    weight_decay=1e-4,
    val_ratio=0.2,
    threshold=0.5,
    random_seed=42,
    channels=64,
    dropout=0.3,
)


Project root: /home/jue/760
Data root: /home/jue/760/dataset


In [75]:
set_seed(config.random_seed)
device = get_device()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using device: {device}")
print("Loading feature arrays...")
x_all = load_features(DATA_ROOT, TRAIN_FEATURE_PATHS)
y_all = load_array(DATA_ROOT / "database_labels_81_big.npy")
x_test = load_features(DATA_ROOT, TEST_FEATURE_PATHS)
y_test = load_array(DATA_ROOT / "database_labels_81_test.npy")

if x_all.shape[0] != y_all.shape[0]:
    raise ValueError(f"Train features/labels mismatch: {x_all.shape[0]} vs {y_all.shape[0]}")
if x_test.shape[0] != y_test.shape[0]:
    raise ValueError(f"Test features/labels mismatch: {x_test.shape[0]} vs {y_test.shape[0]}")

idx_train, idx_val = train_test_split(
    np.arange(len(y_all)),
    test_size=config.val_ratio,
    random_state=config.random_seed,
    shuffle=True,
)

train_loader = DataLoader(
    FeatureDataset(x_all[idx_train], y_all[idx_train]),
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
val_loader = DataLoader(
    FeatureDataset(x_all[idx_val], y_all[idx_val]),
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
test_loader = DataLoader(
    FeatureDataset(x_test, y_test),
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)

model = ResNet6Features(
    input_dim=x_all.shape[1],
    num_classes=y_all.shape[1],
    channels=config.channels,
    dropout=config.dropout,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
criterion = nn.BCEWithLogitsLoss()

best_map = -1.0
history: list[dict[str, float]] = []
checkpoint_path = OUTPUT_DIR / "resnet6_features_best.pt"

print(f"Train samples: {len(idx_train)} | Val samples: {len(idx_val)} | Test samples: {len(y_test)}")
print(f"Input dim: {x_all.shape[1]} | Classes: {y_all.shape[1]}")

for epoch in range(1, config.epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate(model, val_loader, device, config.threshold)
    row = {"epoch": epoch, "train_loss": float(train_loss), **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)

    if val_metrics["mAP"] > best_map:
        best_map = val_metrics["mAP"]
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "config": asdict(config),
                "input_dim": x_all.shape[1],
                "num_classes": y_all.shape[1],
                "best_val_metrics": val_metrics,
            },
            checkpoint_path,
        )

    print(
        f"Epoch {epoch:03d}/{config.epochs} "
        f"loss={train_loss:.4f} "
        f"val_mAP={val_metrics['mAP']:.4f} "
        f"val_micro_f1={val_metrics['micro_f1']:.4f} "
        f"val_macro_f1={val_metrics['macro_f1']:.4f}"
    )

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
test_metrics = evaluate(model, test_loader, device, config.threshold)

save_json(OUTPUT_DIR / "training_history.json", history)
save_json(OUTPUT_DIR / "test_metrics.json", test_metrics)
save_json(OUTPUT_DIR / "config.json", asdict(config))
print(f"Best checkpoint: {checkpoint_path}")
print(f"Test metrics: {test_metrics}")


Using device: cuda
Loading feature arrays...
Train samples: 154987 | Val samples: 38747 | Test samples: 2100
Input dim: 1134 | Classes: 81


/home/jue/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch 001/35 loss=0.1180 val_mAP=0.0526 val_micro_f1=0.0659 val_macro_f1=0.0081


/home/jue/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch 002/35 loss=0.0938 val_mAP=0.0603 val_micro_f1=0.3200 val_macro_f1=0.0334


...

/home/jue/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch 034/35 loss=0.0733 val_mAP=0.1767 val_micro_f1=0.4678 val_macro_f1=0.1256


/home/jue/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch 035/35 loss=0.0733 val_mAP=0.1787 val_micro_f1=0.4850 val_macro_f1=0.1280
Best checkpoint: /home/jue/760/ResNet/resnet6_feature_runs/resnet6_features_best.pt
Test metrics: {'mAP': 0.20227795594056192, 'micro_f1': 0.4689043274462208, 'macro_f1': 0.11690476217861387}


/home/jue/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/jue/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/jue/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/jue/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/jue/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
